# CAD font character graph lab

Builds a planar `CharGraph` for every `source="cad_font"` character in a `scripts/label/CAD_font_label.py` output file, prints per-character debug statistics, and visualizes each graph overlaid on the real labelled PDF page.

This notebook implements steps 2-3 of `docs/cad_font_vector_recognition.md` only (step 1 -- the label file itself -- is built by `CAD_font_label.py`, not here). Structural conventions (config-cell-first, path bootstrap, `path_signature`-based label resolution) follow `rastervec/notebooks/similarity_single_line_cad_text.ipynb`; that notebook's own matching algorithm (a flat delta-chain) is not reused here -- only its layout.

## 0 - Config

In [ ]:
from pathlib import Path

LABEL_JSON_PATH = None   # required: CAD_font_label.py output json
                         # (default outputs/labels/<stem>_cad_font.json)
OUTPUT_DIR = None        # None -> alongside LABEL_JSON_PATH, "<stem>_char_graphs"

BEZIER_SAMPLE_COUNT = 5  # fixed points a "c" item is sampled into (2 endpoints + 3 generated)
                         # epsilon (connect/RDP tolerance) is no longer a config knob -- it's
                         # fully data-derived per character (half the shortest post-split segment)

OVERLAY_DPI = 300        # render_vector_cluster raster dpi for visualization
N_VISUALIZE = 12         # how many characters (complexity order) to render

## 1 - Path bootstrap

In [ ]:
import os, sys
_root = os.path.abspath(os.path.join('../..'))
if _root not in sys.path:
    sys.path.append(_root)

## 2 - Load labels + resolve vectors

The ground-truth source is the JSON file `CAD_font_label.py` writes directly -- its own `pdf_path` is used to reopen the source PDF and re-extract vectors per page (the same `path_signature`-based resolution `build_character_bank` and every other labelling tool use).

In [ ]:
from rastervec.Evaluation.Labelling.label_schema import load_labels
from rastervec.P1_Reading_Native.reader import Reader
from rastervec.P1_Reading_Native.vector_extract import extract_vectors

assert LABEL_JSON_PATH, "set LABEL_JSON_PATH"
labels = load_labels(LABEL_JSON_PATH)
assert labels.pdf_path, f"{LABEL_JSON_PATH} has no pdf_path recorded"

cad_entries = [e for e in labels.entries if e.source == "cad_font"]
print(f"{len(cad_entries)} cad_font character label(s), {len(labels.baselines)} baseline(s)")

if OUTPUT_DIR is None:
    OUTPUT_DIR = str(Path(LABEL_JSON_PATH).parent / (Path(LABEL_JSON_PATH).stem + "_char_graphs"))
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

reader = Reader(labels.pdf_path)
pages_needed = sorted({e.page_index for e in cad_entries})
vectors_by_page = {p: extract_vectors(reader.get_page(p)) for p in pages_needed}
entries_by_label_id = {e.label_id: e for e in cad_entries}
baselines_by_id = {b.baseline_id: b for b in labels.baselines}

## 3 - Build the character bank

In [ ]:
from rastervec.Evaluation.CadFont.character_bank import build_character_bank

templates = build_character_bank(
    labels, vectors_by_page,
    bezier_sample_count=BEZIER_SAMPLE_COUNT,
)
print(f"{len(templates)} character template(s) built (of {len(cad_entries)} cad_font entries)")
if len(templates) < len(cad_entries):
    print(
        "note: entries with no resolvable baseline, or unresolved vector_signatures, "
        "are silently skipped by build_character_bank (see its own log warnings above)"
    )

## 4 - Per-character debug stats

Recomputes the flatten/split/dedup/simplify pipeline step-by-step (rather than only the black-box `build_char_graph`) so every intermediate count -- and each character's own dynamically-derived `scale`/`curve_spacing`/`area_tol` -- is available for the table below. Also derives the exact inverse of `to_baseline_relative_vectors`'s rigid transform for the visualization cell below -- rather than re-deriving its internal two-step (rotate, then x-shift-by-its-own-internal-min_x) formula by hand, which is easy to get subtly wrong since the shift amount isn't exposed by the function, we solve for it **empirically**: the overall map page -> baseline-relative is a single rigid transform (rotation `-theta` + some constant translation), so one known point correspondence (the first resolved vector's own first point, before and after `to_baseline_relative_vectors`) is enough to solve for the translation term of the inverse map directly.

In [ ]:
import math
from collections import Counter

from rastervec.Evaluation.CadFont.character_bank import to_baseline_relative_vectors
from rastervec.Evaluation.CadFont.geometry import split_at_intersections, vectors_to_segments
from rastervec.Evaluation.CadFont.graph import build_char_graph
from rastervec.Evaluation.Labelling.label_schema import path_signature
from rastervec.commons.helpers.geometry import item_points, transform_point


def resolve_vectors(entry):
    sig_map = {path_signature(v): v for v in vectors_by_page[entry.page_index]}
    return [sig_map[s] for s in entry.vector_signatures if s in sig_map]


def inverse_baseline_transform(resolved_vectors, tvecs, baseline):
    """theta + translation to map a baseline-relative-frame point back to page
    space: p_page = R(theta) @ p_rel + t_inv. theta is the same magnitude as the
    forward transform's own rotation (that transform uses -theta); t_inv is solved
    from one known (page, baseline-relative) point pair rather than re-deriving
    the forward transform's own internal min_x shift by hand."""
    theta = math.degrees(math.atan2(baseline.direction[1], baseline.direction[0]))
    p_page0 = item_points(resolved_vectors[0].items[0])[0]
    p_rel0 = item_points(tvecs[0].items[0])[0]
    r_rel0 = transform_point(p_rel0, offset=(0.0, 0.0), rotation_deg=theta)
    t_inv = (p_page0[0] - r_rel0[0], p_page0[1] - r_rel0[1])
    return theta, t_inv


def debug_pipeline_for_entry(entry, baseline, resolved_vectors):
    """`build_char_graph` now runs the whole flatten -> split -> epsilon ->
    connect -> RDP pipeline internally (and returns per-character stats
    alongside the graph), so this only needs to recompute the raw/split
    segment counts for display -- not re-derive any tolerance."""
    tvecs = to_baseline_relative_vectors(resolved_vectors, baseline.origin, baseline.direction)
    raw_segs, vertex_groups = vectors_to_segments(tvecs, bezier_sample_count=BEZIER_SAMPLE_COUNT)
    split_segs = split_at_intersections(raw_segs)
    debug_graph, build_stats = build_char_graph(raw_segs, vertex_groups)
    theta, t_inv = inverse_baseline_transform(resolved_vectors, tvecs, baseline)
    return raw_segs, split_segs, debug_graph, build_stats, theta, t_inv


rows = []
debug_by_label_id = {}
for t in templates:
    entry = entries_by_label_id[t.label_id]
    baseline = baselines_by_id[t.baseline_id]
    resolved = resolve_vectors(entry)
    (raw_segs, split_segs, debug_graph, build_stats,
     theta, t_inv) = debug_pipeline_for_entry(entry, baseline, resolved)
    debug_by_label_id[t.label_id] = (resolved, theta, t_inv)
    degrees = debug_graph.degrees()
    # Anchor points (including any synthetic baseline point) are only valid
    # against t.graph -- the graph select_anchor_points actually returned and
    # build_character_bank stored -- never against debug_graph, which never
    # goes through select_anchor_points and so never gains a synthetic node.
    rows.append({
        "label_id": t.label_id[:10],
        "text": t.text,
        "page": entry.page_index,
        "raw_segs": len(raw_segs),
        "split_segs": len(split_segs),
        "nodes": debug_graph.num_nodes(),
        "edges": debug_graph.num_edges(),
        "degree_dist": dict(Counter(degrees)),
        "max_degree": debug_graph.max_degree(),
        "complexity": t.complexity,
        "epsilon": build_stats.epsilon,
        "points_before_rdp": build_stats.points_before_rdp,
        "points_after_rdp": build_stats.points_after_rdp,
        "points_removed": build_stats.points_removed,
        "synthetic_anchor": bool(t.graph.synthetic_anchor_indices),
        "anchors": [(i, t.graph.nodes[i]) for i in t.anchor_node_indices],
    })

for row in rows:
    print(
        f'{row["label_id"]:12} "{row["text"]}"  page={row["page"]}  '
        f'segs: raw={row["raw_segs"]} split={row["split_segs"]}  '
        f'nodes={row["nodes"]} edges={row["edges"]} max_deg={row["max_degree"]}  '
        f'complexity={row["complexity"]:.0f}  degrees={row["degree_dist"]}  '
        f'epsilon={row["epsilon"]:.4f}  '
        f'points removed by RDP: {row["points_removed"]} '
        f'({row["points_before_rdp"]} -> {row["points_after_rdp"]})  '
        f'synthetic_anchor={row["synthetic_anchor"]}  '
        f'anchors={[i for i, _ in row["anchors"]]}'
    )

## 5 - Visualize: graph overlay on the labelled PDF

`CharGraph.nodes` are stored in **baseline-relative frame** (baseline rotated to `y=0`, the character's own leftmost point translated to `x=0` -- per step 2's spec). To draw them on top of a real rendered page/cluster image, we map them back to page space via the `theta`/`t_inv` inverse transform solved in the previous cell, then into that render's own pixel space via `commons.renderer.png.page_points_to_pixel` -- both the background raster (`render_vector_cluster`, the character's own real vectors) and the graph nodes share the same pixel-space convention, so they land in register.

In [ ]:
import matplotlib.pyplot as plt

from rastervec.commons.renderer.png import render_vector_cluster, page_points_to_pixel


for t in templates[:N_VISUALIZE]:
    entry = entries_by_label_id[t.label_id]
    baseline = baselines_by_id[t.baseline_id]
    resolved, theta, t_inv = debug_by_label_id[t.label_id]
    graph = t.graph

    img = render_vector_cluster(resolved, dpi=OVERLAY_DPI, padding=2.0)
    page_pts = [transform_point(n, offset=t_inv, rotation_deg=theta) for n in graph.nodes]
    pixel_pts = page_points_to_pixel(resolved, OVERLAY_DPI, page_pts, padding=2.0)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(img)
    for a, b in graph.edges:
        (x0, y0), (x1, y1) = pixel_pts[a], pixel_pts[b]
        ax.plot([x0, x1], [y0, y1], color="lime", linewidth=1.5)
    degs = graph.degrees()
    xs, ys = zip(*pixel_pts)
    colors = ["red" if i in t.anchor_node_indices else "cyan" for i in range(len(pixel_pts))]
    sizes = [20 + 10 * d for d in degs]
    ax.scatter(xs, ys, c=colors, s=sizes, zorder=3, edgecolors="black", linewidths=0.5)
    ax.set_title(
        f'"{t.text}"  complexity={t.complexity:.0f}  '
        f'nodes={graph.num_nodes()}  edges={graph.num_edges()}  max_deg={graph.max_degree()}'
    )
    ax.axis("off")
    fig.savefig(Path(OUTPUT_DIR) / f"{entry.label_id}_graph.png", dpi=150, bbox_inches="tight")
    plt.show()

## 6 - Save debug stats

In [ ]:
import json

stats_path = Path(OUTPUT_DIR) / "stats.json"
with open(stats_path, "w", encoding="utf-8") as f:
    json.dump(rows, f, indent=2, default=str)
print(f"wrote {stats_path}")